# Testing with pytest

A function that splits podcast transcripts into pieces for search keeps losing the last sentence. You will catch the bug with a small test runner you write yourself, then with pytest: parametrized cases, the assert report and fixtures. Then you break the fix on purpose and repair it.

About 10 minutes. Run the cells from top to bottom.

## 1. Install pytest

Colab already has pytest, but maybe another version. This installs the version the lesson used, so your output matches it. Look for the version number at the end.

In [1]:
%pip install -q pytest==9.2.0
!python -m pytest --version

Note: you may need to restart the kernel to use updated packages.


pytest 9.2.0


## 2. The splitter, as the search team found it

The next cell saves the function to a file, `splitter.py`, so that pytest can import it later. It cuts the text at each full stop, then fills pieces of at most `max_chars` characters with whole sentences.

It has a bug. Do not fix it yet. The cell only prints `Writing splitter.py`.

In [2]:
%%writefile splitter.py
def split_transcript(text, max_chars):
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    pieces, current = [], ""
    for sentence in sentences:
        sentence = sentence + "."
        if current and len(current) + 1 + len(sentence) > max_chars:
            pieces.append(current)
            current = sentence
        else:
            current = (current + " " + sentence).strip()
    return pieces

Writing splitter.py


## 3. The episode

The next cell runs the splitter on a short episode with `max_chars = 80`, prints each piece, and then lists the sentences that are in no piece.

**Look for:** 3 pieces, and one missing sentence: `Thanks for listening.` Nothing crashed. The function quietly returned less than it was given. The cell also saves the episode to `episode.txt` for step 6.

In [3]:
from pathlib import Path
from splitter import split_transcript

episode = ("Welcome back to Rooftop Hives. Today we visit a beekeeper in the city. "
           "She keeps forty hives on a hotel roof. The bees fly up to three kilometres for food. "
           "Next week we talk about winter. Thanks for listening.")
Path("episode.txt").write_text(episode + "\n")

pieces = split_transcript(episode, 80)
for i, piece in enumerate(pieces, 1):
    print(f"piece {i}: {piece}")

sentences = [s.strip() + "." for s in episode.split(".") if s.strip()]
missing = [s for s in sentences if not any(s in p for p in pieces)]
print("sentences in no piece:", missing)

piece 1: Welcome back to Rooftop Hives. Today we visit a beekeeper in the city.
piece 2: She keeps forty hives on a hotel roof.
piece 3: The bees fly up to three kilometres for food. Next week we talk about winter.
sentences in no piece: ['Thanks for listening.']


## 4. A test runner from scratch

A test is an input plus the output you decided on beforehand, checked by a program. Here are the lesson's five cases for `max_chars = 30`: empty text, one sentence, two that fit, one that spills over, one that is too long. `run_cases` calls the splitter on each input and compares with `==`. In Python, `==` on two lists compares the items, not the references.

**Look for:** `PASS ''` for the empty case, then 4 lines starting with `FAIL`. Each one shows a list that is one piece short. The last line says `failed: 4`.

In [4]:
def run_cases(cases):
    failed = 0
    for text, expected in cases:
        got = split_transcript(text, 30)
        if got == expected:
            print("PASS", repr(text))
        else:
            failed += 1
            print("FAIL", repr(text),
                  "got", got)
    return failed

CASES = [
    ("", []),
    ("Welcome back.", ["Welcome back."]),
    ("Welcome back. Today: bees.", ["Welcome back. Today: bees."]),
    ("Welcome back. Today: bees. Thanks for listening.", ["Welcome back. Today: bees.", "Thanks for listening."]),
    ("Our guest keeps forty hives on a roof. Hi.", ["Our guest keeps forty hives on a roof.", "Hi."]),
]

print("failed:", run_cases(CASES))

PASS ''
FAIL 'Welcome back.' got []
FAIL 'Welcome back. Today: bees.' got []
FAIL 'Welcome back. Today: bees. Thanks for listening.' got ['Welcome back. Today: bees.']
FAIL 'Our guest keeps forty hives on a roof. Hi.' got ['Our guest keeps forty hives on a roof.']
failed: 4


## 5. The same cases as a pytest file

pytest runs every function whose name starts with `test` in files named `test_*.py`. `@pytest.mark.parametrize` runs `test_split` once for each tuple in the list. Inside there is a plain `assert`, with what the function returned on the left and the expected list on the right.

The next cell only saves the file.

In [5]:
%%writefile test_splitter.py
import pytest
from splitter import split_transcript

@pytest.mark.parametrize("text, expected", [
    ("", []),
    ("Welcome back.", ["Welcome back."]),
    ("Welcome back. Today: bees.", ["Welcome back. Today: bees."]),
    ("Welcome back. Today: bees. Thanks for listening.", ["Welcome back. Today: bees.", "Thanks for listening."]),
    ("Our guest keeps forty hives on a roof. Hi.", ["Our guest keeps forty hives on a roof.", "Hi."]),
])
def test_split(text, expected):
    assert split_transcript(text, 30) == expected

Writing test_splitter.py


## 6. Episode tests with a fixture

Three tests check the whole episode: no piece too long, the last sentence kept, no word lost. They all need the transcript. `episode` is a **fixture**: a function marked `@pytest.fixture`. A test asks for it by naming it as a parameter, and pytest passes in what it returns.

The next cell only saves the file.

In [6]:
%%writefile test_episode.py
from pathlib import Path
import pytest
from splitter import split_transcript

@pytest.fixture
def episode():
    print("loading episode")
    return Path("episode.txt").read_text()

def test_no_piece_too_long(episode):
    assert all(len(p) <= 80 for p in split_transcript(episode, 80))

def test_ends_with_last_sentence(episode):
    assert split_transcript(episode, 80)[-1].endswith("Thanks for listening.")

def test_no_words_lost(episode):
    assert " ".join(split_transcript(episode, 80)).split() == episode.split()

Writing test_episode.py


## 7. Run pytest on the buggy splitter

`-q` makes the report short. The `!` at the start runs a shell command from the notebook.

**Look for**, in the first report:
- the first line is `.FFFF`: one character per test, a dot for a pass and `F` for a failure. One test function ran 5 times.
- in the failure for `Welcome back. Today: bees. Thanks for listening.`, the line `E       Right contains one more item: 'Thanks for listening.'`. Left is what the function returned, Right is your expected list. The expected list has one more item, so the function lost one.
- the last line: `4 failed, 1 passed`.

In the second report (the episode tests): `2 failed, 1 passed`. The length test passes even though text is lost. A passing test only checks what it checks.

In [7]:
!python -m pytest -q test_splitter.py
!python -m pytest -q test_episode.py

.

FFFF                                                                    [100%]
=================================== FAILURES ===================================
_____________________ test_split[Welcome back.-expected1] ______________________

text = 'Welcome back.', expected = ['Welcome back.']

    @pytest.mark.parametrize("text, expected", [
        ("", []),
        ("Welcome back.", ["Welcome back."]),
        ("Welcome back. Today: bees.", ["Welcome back. Today: bees."]),
        ("Welcome back. Today: bees. Thanks for listening.", ["Welcome back. Today: bees.", "Thanks for listening."]),
        ("Our guest keeps forty hives on a roof. Hi.", ["Our guest keeps forty hives on a roof.", "Hi."]),
    ])
    def test_split(text, expected):
>       assert split_transcript(text, 30) == expected
E       AssertionError: assert [] == ['Welcome back.']
E         
E         Right contains one more item: 'Welcome back.'
E         Use -v to get more diff

test_splitter.py:12: AssertionError
___

.F

F                                                                      [100%]
=================================== FAILURES ===================================
_________________________ test_ends_with_last_sentence _________________________

episode = 'Welcome back to Rooftop Hives. Today we visit a beekeeper in the city. She keeps forty hives on a hotel roof. The bees fly up to three kilometres for food. Next week we talk about winter. Thanks for listening.\n'

    def test_ends_with_last_sentence(episode):
>       assert split_transcript(episode, 80)[-1].endswith("Thanks for listening.")
E       AssertionError: assert False
E        +  where False = <built-in method endswith of str object at 0x10adf7b30>('Thanks for listening.')
E        +    where <built-in method endswith of str object at 0x10adf7b30> = 'The bees fly up to three kilometres for food. Next week we talk about winter.'.endswith

test_episode.py:14: AssertionError
---------------------------- Captured stdout setup ------

## 8. The quick fix

A teammate adds one line after the loop, `pieces.append(current)`, and the cell saves the new splitter. Then pytest runs the five cases again.

**Look for:** the first line is now `F....`. The empty case fails: `assert [''] == []` and `Left contains one more item: ''`. For empty text the loop never runs, `current` is still `""`, and the new line appends it anyway. The last line says `1 failed, 4 passed`.

In [8]:
%%writefile splitter.py
def split_transcript(text, max_chars):
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    pieces, current = [], ""
    for sentence in sentences:
        sentence = sentence + "."
        if current and len(current) + 1 + len(sentence) > max_chars:
            pieces.append(current)
            current = sentence
        else:
            current = (current + " " + sentence).strip()
    pieces.append(current)
    return pieces

Overwriting splitter.py


In [9]:
!python -m pytest -q test_splitter.py

F....                                                                    [100%]
=================================== FAILURES ===================================
____________________________ test_split[-expected0] ____________________________

text = '', expected = []

    @pytest.mark.parametrize("text, expected", [
        ("", []),
        ("Welcome back.", ["Welcome back."]),
        ("Welcome back. Today: bees.", ["Welcome back. Today: bees."]),
        ("Welcome back. Today: bees. Thanks for listening.", ["Welcome back. Today: bees.", "Thanks for listening."]),
        ("Our guest keeps forty hives on a roof. Hi.", ["Our guest keeps forty hives on a roof.", "Hi."]),
    ])
    def test_split(text, expected):
>       assert split_transcript(text, 30) == expected
E       AssertionError: assert [''] == []
E         
E         Left contains one more item: ''
E         Use -v to get more diff

test_splitter.py:12: AssertionError
=========================== short test summary info =====

## 9. The right fix, and a second fixture scope

The next cell saves the fixed splitter: it adds the last piece only when there is one (`if current:`).

In [10]:
%%writefile splitter.py
def split_transcript(text, max_chars):
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    pieces, current = [], ""
    for sentence in sentences:
        sentence = sentence + "."
        if current and len(current) + 1 + len(sentence) > max_chars:
            pieces.append(current)
            current = sentence
        else:
            current = (current + " " + sentence).strip()
    if current:
        pieces.append(current)
    return pieces

Overwriting splitter.py


The next cell saves a copy of the episode tests that differs in one line: `@pytest.fixture(scope="module")`. With module scope, pytest makes the value once for the whole file and gives the same object to every test in it. The default, function scope, makes a fresh value for each test.

In [11]:
%%writefile test_episode_module.py
from pathlib import Path
import pytest
from splitter import split_transcript

@pytest.fixture(scope="module")
def episode():
    print("loading episode")
    return Path("episode.txt").read_text()

def test_no_piece_too_long(episode):
    assert all(len(p) <= 80 for p in split_transcript(episode, 80))

def test_ends_with_last_sentence(episode):
    assert split_transcript(episode, 80)[-1].endswith("Thanks for listening.")

def test_no_words_lost(episode):
    assert " ".join(split_transcript(episode, 80)).split() == episode.split()

Writing test_episode_module.py


## 10. Run everything on the fixed splitter

pytest hides what tests print unless you add `-s`. The two episode runs use `-s`, so you can count how often the fixture runs.

**Look for:**
- the five cases: `.....` and `5 passed`.
- function scope: `loading episode` printed 3 times, once before each test, then `3 passed`.
- module scope: `loading episode` printed once, then `...` and `3 passed`.

In [12]:
!python -m pytest -q test_splitter.py
!python -m pytest -q -s test_episode.py
!python -m pytest -q -s test_episode_module.py

.....                                                                    [100%]
5 passed in 0.01s


loading episode


.loading episode
.loading episode
.
3 passed in 0.01s


loading episode
...
3 passed in 0.01s


## Now change it

Each cell below has one line marked `EDIT`. Change it, run the cell again, and check that the output changes the way the sentence says.

**Experiment 1: the limit.** The cell splits the three-sentence case with the fixed splitter and prints the length of each piece. Change `MAX_CHARS` to `60`: all three sentences fit in one piece of 48 characters. Then try `10`: you get three pieces, each longer than 10, because the splitter never cuts a sentence.

In [13]:
import importlib, splitter
importlib.reload(splitter)  # load the fixed splitter you saved last

MAX_CHARS = 30  # EDIT: try 60, then 10
for piece in splitter.split_transcript("Welcome back. Today: bees. Thanks for listening.", MAX_CHARS):
    print(len(piece), repr(piece))

26 'Welcome back. Today: bees.'
21 'Thanks for listening.'


**Experiment 2: write your own case.** The expected list on the marked line is wrong on purpose, so `run_cases` prints `FAIL` and `failed: 1`. Work out the right output from the rules (8 characters, limit 30) and fix the line. Then it prints `PASS` and `failed: 0`. A test is only as good as the expected value you write down.

In [14]:
split_transcript = splitter.split_transcript  # run_cases now uses the fixed splitter

my_cases = [
    ("Hi. Bye.", ["Hi.", "Bye."]),  # EDIT: fix the expected list
]
print("failed:", run_cases(my_cases))

FAIL 'Hi. Bye.' got ['Hi. Bye.']
failed: 1


## What you should see

- `pytest -q` on the first splitter prints `.FFFF` and ends with `4 failed, 1 passed`.
- The failure for the three-sentence case says `Right contains one more item: 'Thanks for listening.'`
- With `-s`, the `episode` fixture prints "loading episode" 3 times; with `scope="module"`, once.
- The 3 episode tests against the first splitter end with `2 failed, 1 passed`: the length test passes even though text is lost.
- After the `if current:` fix, the five cases end with `5 passed`.